When i ran the # CELL 14: Training Visualization & Checkpoint Utilities
================================================================================
  TRAINING VISUALIZATION
================================================================================


📊 Training Metrics Summary:
   Min Train Loss:  0.0513 (Epoch 25)
   Min Val Loss:    0.2278 (Epoch 25)
   Max Train Acc:   97.15% (Epoch 25)
   Max Val Acc:     92.39% (Epoch 22)
   Total GPU Mem:   5281 MB peak

✅ Good generalization: Train-Val gap = 4.8%

✅ Training curves saved to: /kaggle/working/MobilePlantViT/training_curves.png

============================================================
  CHECKPOINT VERIFICATION
============================================================

📁 Best Model Checkpoint:

🔍 Verifying checkpoint: /kaggle/working/MobilePlantViT/checkpoints/MobilePlantViT-Base_best.pth
------------------------------------------------------------
   ❌ Verification failed: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([scalar])` or the `torch.serialization.safe_globals([scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.

📁 Last Model Checkpoint:
   File size: 14.87 MB
   ✅ Checkpoint exists

✅ Checkpoint info saved to: /kaggle/working/MobilePlantViT/checkpoint_info.json

✅ Checkpoint utilities defined:
   • load_checkpoint(path) - Load model from checkpoint
   • verify_checkpoint(path) - Verify checkpoint integrity
   • resume_training(path, model, optimizer) - Resume training

   There i found verification failed. Help me fix that

The issue is that PyTorch 2.6+ changed the default behavior of `torch.load()` to `weights_only=True` for security reasons, but your checkpoint contains numpy scalar objects in the training history that aren't allowed by default.

Here's the fix - update the `load_checkpoint` and `verify_checkpoint` functions in CELL 14 to explicitly set `weights_only=False`:



In [ ]:
// Update the load_checkpoint function in CELL 14

def load_checkpoint(checkpoint_path, device=None):
    """
    Load a comprehensive checkpoint and reconstruct the model.
    
    Args:
        checkpoint_path: Path to the checkpoint file
        device: Device to load model to (auto-detected if None)
    
    Returns:
        Dictionary containing:
        - 'model': Reconstructed and loaded model
        - 'optimizer': Optimizer with loaded state (if training_mode=True)
        - 'scheduler': Scheduler with loaded state (if exists)
        - 'epoch': Last completed epoch
        - 'best_val_acc': Best validation accuracy
        - 'history': Training history
        - 'config': Model configuration
        - 'metadata': Additional metadata
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"📂 Loading checkpoint from: {checkpoint_path}")
    
    # Load checkpoint with weights_only=False to allow numpy arrays in history
    # This is safe for checkpoints we created ourselves
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    # Extract metadata
    model_type = checkpoint.get('model_type', 'unknown')
    model_name = checkpoint.get('model_name', 'Unknown Model')
    num_classes = checkpoint.get('num_classes', 38)
    
    print(f"   Model: {model_name}")
    print(f"   Type: {model_type}")
    print(f"   Classes: {num_classes}")
    print(f"   Best Val Acc: {checkpoint.get('best_val_acc', 0)*100:.2f}%")
    print(f"   Epoch: {checkpoint.get('epoch', 0) + 1}")
    
    # Reconstruct model based on type
    if model_type == 'mobileplant_vit':
        model_config = checkpoint.get('model_config', None)
        
        if model_config is not None:
            # Reconstruct from saved config
            print(f"   Reconstructing from saved config...")
            config = MobilePlantViTConfig(**model_config)
            loaded_model = MobilePlantViT(config)
        else:
            # Fallback to base variant
            print(f"   ⚠️  No config found, using base variant")
            loaded_model = mobileplant_vit_base(num_classes=num_classes)
    
    elif model_type == 'mobilenet_v2':
        loaded_model = models.mobilenet_v2(weights=None)
        loaded_model.classifier[1] = nn.Linear(
            loaded_model.classifier[1].in_features, 
            num_classes
        )
    else:
        raise ValueError(f"Unknown model type: {model_type}")
    
    # Load state dict
    loaded_model.load_state_dict(checkpoint['model_state_dict'])
    loaded_model = loaded_model.to(device)
    loaded_model.eval()
    
    print(f"   ✅ Model loaded successfully!")
    
    # Prepare result
    result = {
        'model': loaded_model,
        'epoch': checkpoint.get('epoch', 0),
        'best_val_acc': checkpoint.get('best_val_acc', 0),
        'best_val_loss': checkpoint.get('best_val_loss', float('inf')),
        'history': checkpoint.get('history', {}),
        'config': checkpoint.get('model_config', {}),
        'class_names': checkpoint.get('class_names', []),
        'num_classes': num_classes,
        'training_config': checkpoint.get('training_config', {}),
        'timestamp': checkpoint.get('timestamp', 'unknown'),
        'is_best': checkpoint.get('is_best', False),
    }
    
    # Include optimizer state if available
    if 'optimizer_state_dict' in checkpoint:
        result['optimizer_state_dict'] = checkpoint['optimizer_state_dict']
    
    # Include scheduler state if available
    if 'scheduler_state_dict' in checkpoint:
        result['scheduler_state_dict'] = checkpoint['scheduler_state_dict']
    
    return result


def verify_checkpoint(checkpoint_path):
    """
    Verify checkpoint integrity and display contents.
    
    Args:
        checkpoint_path: Path to checkpoint file
    
    Returns:
        True if checkpoint is valid, False otherwise
    """
    print(f"\n🔍 Verifying checkpoint: {checkpoint_path}")
    print("-" * 60)
    
    try:
        # Load with weights_only=False since our checkpoints contain numpy arrays
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        
        # Required keys
        required_keys = ['model_state_dict', 'model_type', 'num_classes']
        optional_keys = [
            'model_name', 'model_config', 'optimizer_state_dict',
            'scheduler_state_dict', 'epoch', 'best_val_acc', 'best_val_loss',
            'history', 'class_names', 'training_config', 'timestamp', 'is_best'
        ]
        
        # Check required keys
        missing_required = [k for k in required_keys if k not in checkpoint]
        if missing_required:
            print(f"   ❌ Missing required keys: {missing_required}")
            return False
        
        # Display checkpoint contents
        print(f"   ✅ Required keys present")
        print(f"\n   📋 Checkpoint Contents:")
        
        for key in checkpoint.keys():
            value = checkpoint[key]
            if isinstance(value, dict):
                if key == 'model_state_dict':
                    print(f"      • {key}: {len(value)} parameters")
                elif key == 'history':
                    epochs = len(value.get('train_loss', []))
                    print(f"      • {key}: {epochs} epochs recorded")
                else:
                    print(f"      • {key}: dict with {len(value)} entries")
            elif isinstance(value, list):
                print(f"      • {key}: list with {len(value)} items")
            elif isinstance(value, (int, float, str, bool)):
                print(f"      • {key}: {value}")
            else:
                print(f"      • {key}: {type(value).__name__}")
        
        # Verify model can be loaded
        print(f"\n   🔄 Testing model reconstruction...")
        result = load_checkpoint(checkpoint_path)
        
        # Quick inference test
        test_input = torch.randn(1, 3, 224, 224).to(next(result['model'].parameters()).device)
        with torch.no_grad():
            output = result['model'](test_input)
        
        print(f"   ✅ Model inference test passed")
        print(f"      Output shape: {tuple(output.shape)}")
        
        return True
        
    except Exception as e:
        print(f"   ❌ Verification failed: {e}")
        return False


def resume_training(checkpoint_path, model, optimizer, scheduler=None, device=None):
    """
    Resume training from a checkpoint.
    
    Args:
        checkpoint_path: Path to checkpoint file
        model: Model instance (must match checkpoint architecture)
        optimizer: Optimizer instance
        scheduler: Optional scheduler instance
        device: Device to use
    
    Returns:
        Dictionary with loaded state and starting epoch
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"📂 Resuming training from: {checkpoint_path}")
    
    # Load with weights_only=False for our checkpoints
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    
    # Load optimizer state
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # Load scheduler state if available
    if scheduler is not None and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    # Get training state
    start_epoch = checkpoint.get('epoch', 0) + 1
    best_val_acc = checkpoint.get('best_val_acc', 0)
    history = checkpoint.get('history', {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'lr': [], 'epoch_time': [], 'gpu_memory_mb': []
    })
    
    print(f"   ✅ Resumed from epoch {start_epoch}")
    print(f"   Best Val Acc so far: {best_val_acc*100:.2f}%")
    
    return {
        'model': model,
        'optimizer': optimizer,
        'scheduler': scheduler,
        'start_epoch': start_epoch,
        'best_val_acc': best_val_acc,
        'history': history,
    }



The key change is adding `weights_only=False` to all `torch.load()` calls:



In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)



This is safe because:
1. You created these checkpoints yourself during training
2. The numpy scalars in the history (like GPU memory values) are the reason for the error
3. You're loading from your own working directory, not untrusted sources

After making this change, re-run CELL 14 and the verification should pass.

now when i ran the # CELL 15: Test Set Evaluation
================================================================================
  TEST SET EVALUATION
================================================================================
📂 Loading best model checkpoint...
📂 Loading checkpoint from: /kaggle/working/MobilePlantViT/checkpoints/MobilePlantViT-Base_best.pth
---------------------------------------------------------------------------
UnpicklingError                           Traceback (most recent call last)
/tmp/ipykernel_47/2913362363.py in <cell line: 0>()
     14 
     15 # Use the checkpoint loading utility
---> 16 checkpoint_result = load_checkpoint(best_model_path, device=device)
     17 eval_model = checkpoint_result['model']
     18 eval_model.eval()

/tmp/ipykernel_47/1612941012.py in load_checkpoint(checkpoint_path, device)
    150 
    151     # Load checkpoint
--> 152     checkpoint = torch.load(checkpoint_path, map_location=device)
    153 
    154     # Extract metadata

/usr/local/lib/python3.11/dist-packages/torch/serialization.py in load(f, map_location, pickle_module, weights_only, mmap, **pickle_load_args)
   1468                         )
   1469                     except pickle.UnpicklingError as e:
-> 1470                         raise pickle.UnpicklingError(_get_wo_message(str(e))) from None
   1471                 return _load(
   1472                     opened_zipfile,

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([scalar])` or the `torch.serialization.safe_globals([scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.

The issue is that CELL 14 defines the `load_checkpoint` function, but you need to re-run it after making the fix. The error shows that the `load_checkpoint` function in memory still has the old code without `weights_only=False`.

Looking at your notebook, I can see that CELL 14 already has the updated `load_checkpoint` function with `weights_only=False`. However, it seems like the cell wasn't re-run after the update, or there's a version mismatch.

Let me check the exact line causing the issue. The error points to line 152 in the `load_checkpoint` function, which shows:



In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device)



This is **missing** the `weights_only=False` parameter. Your CELL 14 code in the notebook shows the correct version, but you need to **re-run CELL 14** to update the function definition in memory.

**Quick Fix - Re-run CELL 14:**

Simply re-run CELL 14 (Training Visualization & Checkpoint Utilities) which contains the corrected `load_checkpoint` function with `weights_only=False`:



In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)



**Alternative - If you want to fix it directly in CELL 15:**

Add this code at the beginning of CELL 15 before calling `load_checkpoint`:



In [ ]:
# CELL 15: Test Set Evaluation - Quick fix for checkpoint loading

# Add this at the very beginning of CELL 15, before the existing code:

# Override load_checkpoint to fix weights_only issue
def load_checkpoint(checkpoint_path, device=None):
    """Load checkpoint with weights_only=False for compatibility."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"📂 Loading checkpoint from: {checkpoint_path}")
    
    # Load with weights_only=False (safe for our own checkpoints)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    model_type = checkpoint.get('model_type', 'unknown')
    model_name_loaded = checkpoint.get('model_name', 'Unknown Model')
    num_classes = checkpoint.get('num_classes', 38)
    
    print(f"   Model: {model_name_loaded}")
    print(f"   Type: {model_type}")
    print(f"   Classes: {num_classes}")
    print(f"   Best Val Acc: {checkpoint.get('best_val_acc', 0)*100:.2f}%")
    print(f"   Epoch: {checkpoint.get('epoch', 0) + 1}")
    
    if model_type == 'mobileplant_vit':
        model_config = checkpoint.get('model_config', None)
        if model_config is not None:
            print(f"   Reconstructing from saved config...")
            config = MobilePlantViTConfig(**model_config)
            loaded_model = MobilePlantViT(config)
        else:
            print(f"   ⚠️  No config found, using base variant")
            loaded_model = mobileplant_vit_base(num_classes=num_classes)
    elif model_type == 'mobilenet_v2':
        loaded_model = models.mobilenet_v2(weights=None)
        loaded_model.classifier[1] = nn.Linear(loaded_model.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Unknown model type: {model_type}")
    
    loaded_model.load_state_dict(checkpoint['model_state_dict'])
    loaded_model = loaded_model.to(device)
    loaded_model.eval()
    print(f"   ✅ Model loaded successfully!")
    
    return {
        'model': loaded_model,
        'epoch': checkpoint.get('epoch', 0),
        'best_val_acc': checkpoint.get('best_val_acc', 0),
        'best_val_loss': checkpoint.get('best_val_loss', float('inf')),
        'history': checkpoint.get('history', {}),
        'config': checkpoint.get('model_config', {}),
        'class_names': checkpoint.get('class_names', []),
        'num_classes': num_classes,
    }

# Now the rest of CELL 15 continues as normal...



**Recommended Action:**
1. Go back to **CELL 14** 
2. **Re-run CELL 14** (this will update the `load_checkpoint` function in memory)
3. Then run **CELL 15** again

The `load_checkpoint` function in CELL 14 already has the correct fix - it just needs to be executed again to update the function definition in the Python runtime.